# Simple Object Detection in PyTorch

This lab will walk you through how to use object detection models available in [torchvision](https://pytorch.org/vision/stable/models.html#object-detection). In the following sections, you will:

* explore torchvision's object detection models
* load the models in your workspace
* preprocess an image for inference
* run inference on the models and inspect the output

Let's get started!

> This notebook is a PyTorch port of the original TensorFlow Hub lab. The detectors come from `torchvision.models.detection` rather than TF Hub, so they are trained on [COCO](https://cocodataset.org/) (80 classes) instead of Open Images, and the class names you see are COCO's.

## Imports

In [ ]:
import tempfile
from io import BytesIO
from urllib.request import urlopen, Request

import torch
import torchvision
from torchvision.models import detection
from torchvision.transforms.functional import convert_image_dtype
from PIL import Image
from PIL import ImageOps

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

### Choose a model from torchvision

torchvision ships a set of detection models together with their pretrained weights.
- You can see the available ones [here](https://pytorch.org/vision/stable/models.html#object-detection).
- Each model has a matching *weights enum* that carries the checkpoint and its metadata, including the list of class names.
- We selected [Faster R-CNN with a ResNet50 FPN backbone](https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn_v2.html), which is the accurate but slower option.
- You can also modify the following cell to choose the other model we selected, [SSDLite with a MobileNetV3 backbone](https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssdlite320_mobilenet_v3_large.html), which is small and fast.

In [ ]:
# you can switch the commented lines here to pick the other model

# faster r-cnn with a resnet50 fpn backbone
model_name = "fasterrcnn_resnet50_fpn_v2"

# You can choose ssdlite mobilenet version 3 instead and compare the results
#model_name = "ssdlite320_mobilenet_v3_large"

#### Load the model

Next, you'll load the model specified by `model_name`.
- torchvision downloads the pretrained weights the first time a model is used, so this will take a few minutes on the first run.

In [ ]:
model_builders = {
    "fasterrcnn_resnet50_fpn_v2": (detection.fasterrcnn_resnet50_fpn_v2,
                                   detection.FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1),
    "ssdlite320_mobilenet_v3_large": (detection.ssdlite320_mobilenet_v3_large,
                                      detection.SSDLite320_MobileNet_V3_Large_Weights.COCO_V1),
}

builder, weights = model_builders[model_name]
model = builder(weights=weights).to(device)

# the COCO class names, indexed by the label ids the model outputs
class_names = weights.meta["categories"]
print(f"{len(class_names)} classes, first few: {class_names[:5]}")

#### Put the model in inference mode

A TF Hub model exposes *signatures*; a torchvision detection model instead has two modes.
- In training mode it expects images **and** targets, and returns a dictionary of losses.
- In evaluation mode (`model.eval()`) it takes just the images and returns the detections, which is what you want here.

For object detection models, evaluation mode accepts a list of image tensors and outputs one dictionary per image describing the objects detected.

In [ ]:
detector = model.eval()
print(type(detector).__name__)

### download_and_resize_image

This function downloads an image specified by a given "url", pre-processes it, and then saves it to disk.

In [ ]:
def download_and_resize_image(url, new_width=256, new_height=256):
    '''
    Fetches an image online, resizes it and saves it locally.

    Args:
        url (string) -- link to the image
        new_width (int) -- size in pixels used for resizing the width of the image
        new_height (int) -- size in pixels used for resizing the length of the image

    Returns:
        (string) -- path to the saved image
    '''

    # create a temporary file ending with ".jpg"
    _, filename = tempfile.mkstemp(suffix=".jpg")

    # opens the given URL. Wikimedia rejects requests without a User-Agent header
    response = urlopen(Request(url, headers={'User-Agent': 'Mozilla/5.0'}))

    # reads the image fetched from the URL
    image_data = response.read()

    # puts the image data in memory buffer
    image_data = BytesIO(image_data)

    # opens the image
    pil_image = Image.open(image_data)

    # resizes the image. will crop if aspect ratio is different.
    pil_image = ImageOps.fit(pil_image, (new_width, new_height), Image.Resampling.LANCZOS)

    # converts to the RGB colorspace
    pil_image_rgb = pil_image.convert("RGB")

    # saves the image to the temporary file created earlier
    pil_image_rgb.save(filename, format="JPEG", quality=90)

    print("Image downloaded to %s." % filename)

    return filename

### Download and preprocess an image

Now, using `download_and_resize_image` you can get a sample image online and save it locally.
- We've provided a URL for you, but feel free to choose another image to run through the object detector.
- You can use the original width and height of the image but feel free to modify it and see what results you get.

In [ ]:
# You can choose a different URL that points to an image of your choice
image_url = "https://upload.wikimedia.org/wikipedia/commons/f/fb/20130807_dublin014.JPG"

# download the image and use the original height and width
downloaded_image_path = download_and_resize_image(image_url, 3872, 2592)

### run_detector

This function will take in the object detection model `detector` and the path to a sample image, then use this model to detect objects and display its predicted class categories and detection boxes.
- `run_detector` uses `load_img` to convert the image into a tensor.

Note that torchvision returns boxes in **pixels** as `[xmin, ymin, xmax, ymax]`, whereas the TF Hub models returned them normalized as `[ymin, xmin, ymax, xmax]`.

In [ ]:
def load_img(path):
    '''
    Loads a JPEG image and converts it to a tensor.

    Args:
        path (string) -- path to a locally saved JPEG image

    Returns:
        (tensor) -- a uint8 image tensor of shape (3, height, width)
    '''

    # read and decode the file
    img = torchvision.io.read_image(path, mode=torchvision.io.ImageReadMode.RGB)

    return img


def run_detector(detector, path, class_names, device):
    '''
    Runs inference on a local file using an object detection model.

    Args:
        detector (nn.Module) -- a torchvision detection model in eval mode
        path (string) -- path to an image saved locally
        class_names (list of str) -- COCO category names indexed by label id
        device (torch.device) -- device the model runs on
    '''

    # load an image tensor from a local file path
    img = load_img(path)

    # convert to float in the range [0, 1] and move to the device.
    # torchvision detectors take a *list* of image tensors, not a batched tensor
    converted_img = convert_image_dtype(img, torch.float32).to(device)

    # run inference using the model
    with torch.no_grad():
        result = detector([converted_img])[0]

    # save the results in a dictionary
    result = {key: value.cpu().numpy() for key, value in result.items()}

    # print results
    print("Found %d objects." % len(result["scores"]))

    print(result["scores"])
    print([class_names[label] for label in result["labels"]])
    print(result["boxes"])

### Run inference on the image

You can run your detector by calling the `run_detector` function. This will print the number of objects found followed by three lists:

* The detection scores of each object found (i.e. how confident the model is),
* The classes of each object found,
* The bounding boxes of each object

You will see how to overlay this information on the original image in the next sections and in this week's assignment!

In [ ]:
# runs the object detection model and prints information about the objects found
run_detector(detector, downloaded_image_path, class_names, device)